# CCP Literacy Estimation — Step-by-Step Demo (Model 1)

This notebook shows CCP estimation, transitions, CCS, a small MD fit, and a light EM loop.

In [24]:
import numpy as np, pandas as pd, json
from collections import defaultdict

CSV_PATH = 'toy_panel_with_random_x.csv'
BETA=0.96
X_GRID = np.array([0.0,0.25,0.5,0.75,1.0])
J=len(X_GRID)
A_BINS,Y_BINS,AGE_BINS=6,4,4
ALPHA=5.0
H,DRAWS=3,30
K_TYPES=2
EULER_GAMMA=0.5772156649015329

## 1) Load the panel data

**Why**: We need a person–time panel with $(a,y,age,x)$ to build states and choices.


In [25]:
df = pd.read_csv(CSV_PATH)
df.head()

,id,t,age,a,y,x
0,0,0,49,22578.384369,18562.351115,0.986390
1,0,1,50,21927.092309,18448.435478,0.242381
2,0,2,51,21828.647582,18243.992208,0.702797
3,0,3,52,21654.901444,18939.917391,0.695454
4,0,4,53,21921.181994,18889.896102,0.009471


## 2) Define state bins and discretize actions

- CCPs and transitions are nonparametric objects. To estimate them robustly, we need a finite state grid. Quantile bins give balanced cells.
- A discrete action set $J=\{0,0.25,0.5,0.75,1\}$ makes logit CCPs tractable and aligns with Hotz–Miller.

In [26]:
def quantile_bins(series, nbins):
    qs = np.linspace(0,1,nbins+1)
    cuts = series.quantile(qs).values.astype(float)
    for i in range(1,len(cuts)):
        if cuts[i] <= cuts[i-1]:
            cuts[i] = cuts[i-1] + 1e-9
    return cuts

def cut_to_bins(x, cuts):
    return int(np.clip(np.searchsorted(cuts, x, side='right')-1, 0, len(cuts)-2))

def dirichlet_smooth(counts, alpha=5.0):
    counts = np.asarray(counts,float)
    prior = alpha/len(counts)
    return (counts + prior)/(counts.sum()+alpha)

def inclusive_value_from_ccp(P):
    P = np.clip(np.asarray(P,float), 1e-12,1-1e-12)
    return -float(np.log(P[0]))

def softmax(v):
    v = np.asarray(v,float); v = v - np.max(v)
    ex = np.exp(v); return ex/ex.sum()

In [27]:
a_cuts = quantile_bins(df['a'], A_BINS)
y_cuts = quantile_bins(df['y'], Y_BINS)
age_cuts = quantile_bins(df['age'], AGE_BINS)

dfb = df.copy()
dfb['ia'] = dfb['a'].apply(lambda v: cut_to_bins(v, a_cuts))
dfb['iy'] = dfb['y'].apply(lambda v: cut_to_bins(v, y_cuts))
dfb['iage'] = dfb['age'].apply(lambda v: cut_to_bins(v, age_cuts))

def x_to_j(x): 
    return int(np.argmin(np.abs(X_GRID - float(x))))

dfb['j'] = dfb['x'].apply(x_to_j)

dfb = dfb.sort_values(['id','t']).reset_index(drop=True)

for col in ['ia','iy','iage','j']:
    dfb[col+'_next'] = dfb.groupby('id')[col].shift(-1)

dfb = dfb.dropna(subset=['ia_next','iy_next','iage_next']).copy()

for col in ['ia_next','iy_next','iage_next','j_next']:
    dfb[col] = dfb[col].astype(int)

dfb.head()

,id,t,age,a,y,x,ia,iy,iage,j,ia_next,iy_next,iage_next,j_next
0,0,0,49,22578.384369,18562.351115,0.986390,2,0,2,4,2,0,2,1
1,0,1,50,21927.092309,18448.435478,0.242381,2,0,2,1,2,0,2,3
2,0,2,51,21828.647582,18243.992208,0.702797,2,0,2,3,2,1,2,3
3,0,3,52,21654.901444,18939.917391,0.695454,2,1,2,3,2,1,3,0
5,1,0,49,12187.260333,26335.488275,0.384359,0,3,2,2,0,3,2,3


## 3) Estimate CCP $\hat{P}(j \mid s)$

With logit shocks, CCPs invert to value-index differences and yield an inclusive value for the state. They are the data bridge to dynamic values.

In [29]:
J=len(X_GRID)
counts = defaultdict(lambda: np.zeros(J, float))

for _,row in dfb.iterrows():
    s=(int(row['ia']), int(row['iy']), int(row['iage']))
    j=int(row['j'])
    counts[s][j]+=1.0

ccp={}; state_counts={}

for s,cvec in counts.items():
    state_counts[s]=float(cvec.sum())
    ccp[s]=dirichlet_smooth(cvec, alpha=ALPHA)
len(ccp)

90

In [30]:
counts

defaultdict(<function __main__.<lambda>()>,
            {(2, 0, 2): array([1., 2., 0., 1., 1.]),
             (2, 1, 2): array([3., 2., 2., 3., 0.]),
             (0, 3, 2): array([1., 2., 2., 2., 2.]),
             (2, 1, 0): array([1., 6., 3., 2., 0.]),
             (1, 2, 1): array([0., 1., 0., 1., 1.]),
             (2, 2, 1): array([0., 2., 2., 2., 2.]),
             (4, 3, 0): array([1., 1., 1., 3., 3.]),
             (4, 3, 1): array([2., 1., 3., 2., 0.]),
             (0, 0, 1): array([1., 2., 5., 0., 2.]),
             (4, 1, 2): array([1., 0., 3., 2., 3.]),
             (5, 2, 3): array([2., 1., 3., 2., 1.]),
             (3, 0, 2): array([0., 0., 2., 4., 2.]),
             (5, 1, 1): array([1., 1., 1., 2., 0.]),
             (5, 2, 2): array([0., 1., 2., 1., 1.]),
             (5, 1, 2): array([0., 3., 0., 0., 1.]),
             (3, 2, 2): array([1., 1., 2., 2., 3.]),
             (1, 1, 1): array([0., 0., 0., 0., 1.]),
             (4, 3, 2): array([0., 2., 1., 2., 2.]),
  

In [31]:
ccp

{(2, 0, 2): array([0.2, 0.3, 0.1, 0.2, 0.2]),
 (2,
  1,
  2): array([0.26666667, 0.2       , 0.2       , 0.26666667, 0.06666667]),
 (0,
  3,
  2): array([0.14285714, 0.21428571, 0.21428571, 0.21428571, 0.21428571]),
 (2,
  1,
  0): array([0.11764706, 0.41176471, 0.23529412, 0.17647059, 0.05882353]),
 (1, 2, 1): array([0.125, 0.25 , 0.125, 0.25 , 0.25 ]),
 (2,
  2,
  1): array([0.07692308, 0.23076923, 0.23076923, 0.23076923, 0.23076923]),
 (4,
  3,
  0): array([0.14285714, 0.14285714, 0.14285714, 0.28571429, 0.28571429]),
 (4,
  3,
  1): array([0.23076923, 0.15384615, 0.30769231, 0.23076923, 0.07692308]),
 (0,
  0,
  1): array([0.13333333, 0.2       , 0.4       , 0.06666667, 0.2       ]),
 (4,
  1,
  2): array([0.14285714, 0.07142857, 0.28571429, 0.21428571, 0.28571429]),
 (5,
  2,
  3): array([0.21428571, 0.14285714, 0.28571429, 0.21428571, 0.14285714]),
 (3,
  0,
  2): array([0.07692308, 0.07692308, 0.23076923, 0.38461538, 0.23076923]),
 (5, 1, 1): array([0.2, 0.2, 0.2, 0.3, 0.1]),
 (

## 4) Estimate transitions $\hat{p}(s' \mid s,j)$

To compute the continuation under “what if I choos $j$ now?”, we must know how states move given current state+action. We estimate it empirically from the panel.

In [32]:
trans = defaultdict(lambda: defaultdict(float))

for _, row in dfb.iterrows():
    s=(int(row['ia']), int(row['iy']), int(row['iage']))
    j=int(row['j'])
    sp=(int(row['ia_next']), int(row['iy_next']), int(row['iage_next']))
    trans[(s,j)][sp]+=1.0

trans_prob={}
for key,d in trans.items():
    items=list(d.items())
    probs=np.array([v for (_,v) in items], float); probs=probs/probs.sum()
    trans_prob[key]=([sp for (sp,_) in items], probs)

def draw_next_state(s,j):
    key=(s,j)
    if key not in trans_prob:
        cand=[(k,v) for (k,v) in trans_prob.items() if k[0]==s]
        if not cand:
            return s
        sps=[]; ps=[]
        for (_, (sp_list, p_list)) in cand:
            sps+=sp_list; ps+=list(p_list/len(cand))
        ps=np.array(ps,float); ps=ps/ps.sum()
        idx=np.random.choice(len(sps), p=ps)
        return sps[idx]
    sps,ps=trans_prob[key]
    idx=np.random.choice(len(sps), p=ps)
    return sps[idx]

In [33]:
trans_prob

{((2, 0, 2), 4): ([(2, 0, 2)], array([1.])),
 ((2, 0, 2), 1): ([(2, 0, 2), (2, 0, 3)], array([0.5, 0.5])),
 ((2, 0, 2), 3): ([(2, 1, 2)], array([1.])),
 ((2, 1, 2), 3): ([(2, 1, 3), (2, 1, 2), (3, 2, 2)],
  array([0.33333333, 0.33333333, 0.33333333])),
 ((0, 3, 2), 2): ([(0, 3, 2), (1, 3, 2)], array([0.5, 0.5])),
 ((0, 3, 2), 3): ([(0, 3, 2)], array([1.])),
 ((0, 3, 2), 1): ([(0, 3, 2)], array([1.])),
 ((0, 3, 2), 0): ([(0, 3, 3)], array([1.])),
 ((2, 1, 0), 1): ([(1, 2, 1), (2, 1, 0), (2, 1, 1)],
  array([0.16666667, 0.5       , 0.33333333])),
 ((1, 2, 1), 3): ([(2, 2, 1)], array([1.])),
 ((2, 2, 1), 3): ([(2, 2, 1), (1, 1, 1)], array([0.5, 0.5])),
 ((2, 2, 1), 4): ([(2, 2, 1), (3, 2, 1)], array([0.5, 0.5])),
 ((4, 3, 0), 4): ([(4, 3, 0)], array([1.])),
 ((4, 3, 0), 3): ([(4, 3, 1), (5, 3, 0), (4, 3, 0)],
  array([0.33333333, 0.33333333, 0.33333333])),
 ((4, 3, 1), 2): ([(4, 3, 1), (4, 3, 2)], array([0.66666667, 0.33333333])),
 ((4, 3, 1), 0): ([(4, 3, 1)], array([1.])),
 ((0, 0, 1), 

## 5) Compute inclusive values and CCS $\Delta EV (s ; j,0)$

Under logit, the $\textbf{ex-ante value}$ at a state is
$$V(s) = \gamma_E - \ln P(0 \mid s)$$
So we can $\textbf{simulate forward}$ using $\hat{P}$ and $\hat{p}$, summing discounted $\textbf{inclusive values}$, instead of solving Bellman.


In [34]:
states=list(ccp.keys())

def delta_ev_ccs(s,j):
    def path_val(start_s, initial_j):
        vals=[]
        for _ in range(DRAWS):
            s_current=draw_next_state(start_s, initial_j)
            acc=(BETA**1)*inclusive_value_from_ccp(ccp.get(s_current, np.ones(J)/J))
            for h in range(2,H+1):
                P_current=ccp.get(s_current, None)
                if P_current is None:
                    j_draw=0
                else:
                    j_draw=int(np.random.choice(J, p=P_current))
                s_current=draw_next_state(s_current, j_draw)
                acc+=(BETA**h)*inclusive_value_from_ccp(ccp.get(s_current, np.ones(J)/J))
            vals.append(acc)
        return float(np.mean(vals)) if vals else 0.0
    return path_val(s,j) - path_val(s,0)

delta_cache={}
for s in states:
    for j in range(1,J):
        delta_cache[(s,j)] = delta_ev_ccs(s,j)

len(delta_cache)

360

In [35]:
delta_cache

{((2, 0, 2), 1): 0.19720967912856935,
 ((2, 0, 2), 2): 0.5791851632731007,
 ((2, 0, 2), 3): 0.07204103934853556,
 ((2, 0, 2), 4): -0.03557486668826382,
 ((2, 1, 2), 1): 1.2272568069367376,
 ((2, 1, 2), 2): 0.658642390850769,
 ((2, 1, 2), 3): 1.7784544509508855,
 ((2, 1, 2), 4): 0.5138791065076767,
 ((0, 3, 2), 1): 1.4778681083332827,
 ((0, 3, 2), 2): 1.8365690531058068,
 ((0, 3, 2), 3): 1.2467713331719095,
 ((0, 3, 2), 4): 0.5817881110855581,
 ((2, 1, 0), 1): -0.13536077659242185,
 ((2, 1, 0), 2): -0.08463240858545706,
 ((2, 1, 0), 3): -0.3693196559303873,
 ((2, 1, 0), 4): -0.2726579976479391,
 ((1, 2, 1), 1): -0.06176033521758395,
 ((1, 2, 1), 2): -0.1429474643204891,
 ((1, 2, 1), 3): 0.20327197171158407,
 ((1, 2, 1), 4): -0.2063592052167369,
 ((2, 2, 1), 1): 0.6901614253644874,
 ((2, 2, 1), 2): -0.1336645143905697,
 ((2, 2, 1), 3): -0.7540518519683888,
 ((2, 2, 1), 4): 0.12997191644425587,
 ((4, 3, 0), 1): 0.37743169539129706,
 ((4, 3, 0), 2): 0.3669249225588507,
 ((4, 3, 0), 3): 0.0

## 6) Minimum-distance (MD) fit of cost parameters

The HM moment says \textbf{empirical log-odds} must equal $(- \text{expected cost}) + \beta \times \text{CCS difference}.$

With latent types $L$, expected cost is $\textbf{mixture-weighted}$: $\sum_L \pi_L \big(k_0 + k_L + \varphi_L a \big).$


In [36]:
a_mids = np.array([0.5*(a_cuts[b]+a_cuts[b+1]) for b in range(len(a_cuts)-1)], float)

def objective(theta):
    k0, kL_low, kL_high, phi_low, phi_high, pi_low = theta
    pi=np.array([pi_low, max(1e-6, 1.0-pi_low)]); pi=pi/np.sum(pi)
    loss=0.0; wsum=0.0
    for s in states:
        ia,iy,iage = s
        P = ccp[s]
        a_mid = a_mids[ia]
        costs = np.array([k0 + kL_low + phi_low*a_mid, k0 + kL_high + phi_high*a_mid])
        exp_cost = float(np.sum(pi * costs))
        for j in range(1,J):
            logodds = float(np.log(P[j]) - np.log(P[0]))
            lam = -exp_cost + BETA * delta_cache[(s,j)]
            w = max(state_counts.get(s,1.0), 1.0)
            loss += (logodds - lam)**2 * w
            wsum += w
    return loss / max(wsum,1.0)

theta0 = np.array([20.0, 20.0, 5.0, 0.01, 0.005, 0.5], float)
rng = np.random.default_rng(0)
best_theta = theta0.copy(); best_val = objective(best_theta)
for it in range(200):
    cand = best_theta + rng.normal(0, [2,2,2,0.001,0.001,0.05], size=6)
    val = objective(cand)
    if val < best_val:
        best_theta, best_val = cand, val
best_val, best_theta

(7.9771916031685945,
 array([ 2.62685042e-01,  2.99549541e+01, -1.21102124e+00, -3.28823248e-03,
         7.04896717e-04,  2.12771239e-01]))

## 7) EM: infer laten literacy (personal level)

The HM moment says \textbf{empirical log-odds} must equal $(- \text{expected cost}) + \beta \times \text{CCS difference}.$

With latent types $L$, expected cost is $\textbf{mixture-weighted}$: $\sum_L \pi_L \big(k_0 + k_L + \varphi_L a \big).$


In [37]:
def type_ccp(theta):
    k0, kL_low, kL_high, phi_low, phi_high, pi_low = theta
    def ccps_at_state(s, L):
        ia,_,_ = s
        a_mid = a_mids[ia]
        if L==0:
            kL, phi = kL_low, phi_low
        else:
            kL, phi = kL_high, phi_high
        lam = np.zeros(J); lam[0]=0.0
        for j in range(1,J):
            lam[j] = - (k0 + kL + phi*a_mid) + BETA * delta_cache[(s,j)]
        return softmax(lam)
    return ccps_at_state

traj = defaultdict(list)
for _, row in dfb.sort_values(['id','t']).iterrows():
    s_t = (int(row['ia']), int(row['iy']), int(row['iage']))
    j_t = int(row['j'])
    traj[int(row['id'])].append((s_t, j_t))

def em(theta_init, n_iter=2):
    theta = theta_init.copy()
    k0, kL_low, kL_high, phi_low, phi_high, pi_low = theta
    pi = np.array([pi_low, max(1e-6, 1.0-pi_low)]); pi = pi/np.sum(pi)
    for it in range(n_iter):
        ccps = type_ccp(theta)
        omega={}
        for pid, seq in traj.items():
            like=np.zeros(2)
            for L in [0,1]:
                ll=0.0
                for (s_t, j_t) in seq:
                    P = ccps(s_t, L)
                    ll += np.log(max(P[j_t], 1e-12))
                like[L]=np.exp(ll)
            post = pi * like
            s = post.sum()
            omega[pid] = post/s if s>0 else np.array([0.5,0.5])
        # update pi
        mat = np.stack(list(omega.values()))
        pi = mat.mean(axis=0); pi = pi/np.sum(pi)
        # small local search to re-fit costs keeping pi fixed
        def objective_fixed_pi(th):
            th = th.copy()
            th[-1] = pi[0]
            return objective(th)
        best = theta.copy(); best[-1] = pi[0]
        best_val = objective(best)
        for _ in range(100):
            cand = best + np.random.normal(0, [1,1,1,0.0005,0.0005,0.0], size=6)
            val = objective_fixed_pi(cand)
            if val < best_val:
                best, best_val = cand, val
        theta = best; theta[-1] = pi[0]
    return theta, pi, omega

theta_em, pi_em, omega = em(best_theta, n_iter=2)
theta_em, pi_em

(array([ 2.28360209e+00,  2.45559269e+01, -5.25377955e+00,  1.67598934e-03,
        -1.09009610e-03,  3.13153807e-01]),
 array([0.31315381, 0.68684619]))

In [38]:
rows=[]
for pid, post in omega.items():
    rows.append({'id':pid, 'p_low': float(post[0]), 'p_high': float(post[1])})
post_df = pd.DataFrame(rows).sort_values('id').reset_index(drop=True)
post_df.head()

,id,p_low,p_high
0,0,4.993220e-17,1.000000e+00
1,1,2.415025e-05,9.999758e-01
2,2,1.379811e-18,1.000000e+00
3,3,9.999998e-01,2.392811e-07
4,4,2.470785e-25,1.000000e+00


In [40]:
post_df

,id,p_low,p_high
0,0,4.993220e-17,1.000000e+00
1,1,2.415025e-05,9.999758e-01
2,2,1.379811e-18,1.000000e+00
3,3,9.999998e-01,2.392811e-07
4,4,2.470785e-25,1.000000e+00
...,...,...,...
145,145,1.000000e+00,1.975523e-17
146,146,1.160922e-06,9.999988e-01
147,147,1.280283e-25,1.000000e+00
148,148,4.552796e-05,9.999545e-01


In [39]:
summary = {
    'theta_em': list(map(float, theta_em)),
    'pi_em': list(map(float, pi_em)),
    'objective_value': float(objective(theta_em))
}
print(json.dumps(summary, indent=2))

{
  "theta_em": [
    2.2836020913409785,
    24.55592686094956,
    -5.253779547820741,
    0.001675989344067632,
    -0.001090096102197233,
    0.3131538070310215
  ],
  "pi_em": [
    0.3131538070310215,
    0.6868461929689785
  ],
  "objective_value": 6.6061390116660865
}
